In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\heart\quantum_6g_ai")
RESULTS_DIR = PROJECT_ROOT / "results"

TRAIN_PATH = RESULTS_DIR / "train_scaled.csv"
TEST_PATH = RESULTS_DIR / "test_scaled.csv"

print("Project root :", PROJECT_ROOT)
print("Results dir  :", RESULTS_DIR)
print("Train file   :", TRAIN_PATH)
print("Test file    :", TEST_PATH)

Project root : C:\Users\heart\quantum_6g_ai
Results dir  : C:\Users\heart\quantum_6g_ai\results
Train file   : C:\Users\heart\quantum_6g_ai\results\train_scaled.csv
Test file    : C:\Users\heart\quantum_6g_ai\results\test_scaled.csv


In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

FEATURE_COLUMNS = [
    "SINR",
    "RSRP",
    "Latency",
    "Packet_Loss",
    "Throughput",
    "Interference",
    "Resource_Load",
    "Trust_Score",
    "Risk_Score"
]

TARGET_COLUMN = "Label"

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print("Training features:", X_train.shape)
print("Training labels  :", y_train.shape)

print("Testing features :", X_test.shape)
print("Testing labels   :", y_test.shape)

print("\nTraining labels:")
print(y_train.value_counts().sort_index())

print("\nTesting labels:")
print(y_test.value_counts().sort_index())

Training features: (800, 9)
Training labels  : (800,)
Testing features : (200, 9)
Testing labels   : (200,)

Training labels:
Label
0    400
1    400
Name: count, dtype: int64

Testing labels:
Label
0    100
1    100
Name: count, dtype: int64


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    )
}

print("Models:")
for name in models:
    print("-", name)

Models:
- Logistic Regression
- SVM


In [4]:
trained_models = {}

for name, model in models.items():
    print(f"\nTraining: {name}")
    
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    print("Status: completed")


Training: Logistic Regression
Status: completed

Training: SVM
Status: completed


C:\Users\heart\anaconda3\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in trained_models.items():
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC_AUC": roc_auc
    })

metrics_df = pd.DataFrame(results)

print(metrics_df.to_string(index=False))

              Model  Accuracy  Precision  Recall       F1  ROC_AUC
Logistic Regression     0.995   0.990099    1.00 0.995025   1.0000
                SVM     0.975   0.970297    0.98 0.975124   0.9971


In [6]:
from sklearn.metrics import confusion_matrix

for name, model in trained_models.items():
    
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    print("\n" + "=" * 50)
    print(name)
    print("=" * 50)
    print("Confusion Matrix:")
    print(cm)


Logistic Regression
Confusion Matrix:
[[ 99   1]
 [  0 100]]

SVM
Confusion Matrix:
[[97  3]
 [ 2 98]]


In [7]:
METRICS_PATH = RESULTS_DIR / "metrics.csv"

metrics_df.to_csv(
    METRICS_PATH,
    index=False
)

print("Metrics saved successfully:")
print(METRICS_PATH)

Metrics saved successfully:
C:\Users\heart\quantum_6g_ai\results\metrics.csv


In [9]:
print("\n" + "=" * 70)
print("CLASSICAL BASELINE RESULTS")
print("=" * 70)

print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


CLASSICAL BASELINE RESULTS
              Model  Accuracy  Precision  Recall     F1  ROC_AUC
Logistic Regression    0.9950     0.9901  1.0000 0.9950   1.0000
                SVM    0.9750     0.9703  0.9800 0.9751   0.9971


In [10]:
print("\nSaved metrics:")
print(METRICS_PATH)

print("\nCSV verification:")
print(pd.read_csv(METRICS_PATH).to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))


Saved metrics:
C:\Users\heart\quantum_6g_ai\results\metrics.csv

CSV verification:
              Model  Accuracy  Precision  Recall     F1  ROC_AUC
Logistic Regression    0.9950     0.9901  1.0000 0.9950   1.0000
                SVM    0.9750     0.9703  0.9800 0.9751   0.9971
